# What IsGenerative AI? - The Machine That Learned to Create

**Module 3 · Lesson 3.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Two Types of AI - The Judge vs The Chef
- How Does It Work? - The Next-Word Game
- The Universe of Generative AI - It's Not Just ChatGPT
- Why Now? - Three Breakthroughs That Changed Everything
- What Can Go Wrong? - The Limitations You Must Know
- Three Ways to Customize - Prompting → RAG → Fine-Tuning

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai scikit-learn -q google-genai

# Reproducibility - the lesson uses seeded random values throughout

> 🎓 **Analogy**
>
> **Think about how you learned to write essays in school.**
> Your teacher showed you hundreds of essays. You noticed patterns: essays start with a hook, paragraphs have topic sentences, conclusions restate the thesis. After seeing enough examples, you could **write a new essay that nobody had written before** - because you'd internalized the patterns.
> **That's exactly what Generative AI does.** It reads billions of sentences, notices the patterns in how words follow each other, and produces new sentences that follow those same patterns. It never copies. It generates.

> 📦 **Info**
>
> The one-sentence definition
> **Generative AI** = AI that creates new content (text, images, code, audio, video) by learning patterns from existing data and producing novel outputs that follow those patterns.

## Core concepts

---

## 🎯 Concept 1: Two Types of AI - The Judge vs The Chef

### Two Types of AI - The Judge vs The Chef

The most important distinction in all of machine learning.

> 🍲 **Analogy**
>
> **Discriminative AI is a food critic.** You show it a plate. It says: "This is biryani" or "This is pulao." It classifies, labels, judges. But ask it to *cook* biryani? It can't. It only recognizes.
> **Generative AI is a chef.** It studied 10,000 biryani recipes. Now it can cook a *new* biryani that nobody has ever made - maybe with a twist, maybe with a new spice - but it tastes like biryani because it learned what biryani *is*.

**The Judge** - "What IS this?"

- Spam or not spam?

- Cat or dog?

- Fraud or legitimate?

- Positive or negative review?

**The Creator** - "What COULD exist?"

- Write a poem about rain

- Generate a sunset photo

- Compose music

- Write Python code

How each AI processes the same movie review

### See the difference in code

**📝 `discriminative_ai.py`** - *sklearn*

In [ ]:
# =============================================
# DISCRIMINATIVE AI - The Judge
# Input: movie review → Output: "positive" or "negative"
# It classifies. It NEVER creates new text.
# =============================================

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Training data
reviews = [
    "This movie was absolutely fantastic! Loved every minute.",
    "Terrible film. Waste of time and money.",
    "A masterpiece! The acting was incredible.",
    "Boring and predictable. Would not recommend.",
    "One of the best films I've ever seen!",
    "Awful screenplay. The plot made no sense.",
]
labels = ["positive", "negative", "positive", "negative", "positive", "negative"]

# Train
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(reviews)
model = MultinomialNB()
model.fit(X, labels)

# Classify a NEW review
new_review = "The movie was engaging and well-directed."
prediction = model.predict(vectorizer.transform([new_review]))[0]

print(f"Review:  '{new_review}'")
print(f"AI says: {prediction}")
print(f"Can it WRITE a review? ❌ No. It only judges.")
# Output:
# Review:  'The movie was engaging and well-directed.'
# AI says: positive
# Can it WRITE a review? No - it only judges.

**📝 `generative_ai.py Gemini`** - *API*

In [ ]:
# =============================================
# GENERATIVE AI - The Creator
# Input: a prompt → Output: NEW text that never existed
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# Task 1: CREATE a movie review (not classify - CREATE)
resp = client.models.generate_content(model="gemini-2.5-flash", contents=
    "Write a 3-sentence review for a fictional Bollywood "
    "sci-fi film called 'Mars Wala Love'."
)
print("Generated review:")
print(f"  {resp.text}\n")

# Task 2: CREATE code
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Write a Python function to check if a number is prime.")
print("Generated code:")
print(f"  {resp.text}\n")

# Task 3: CREATE an analogy
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Explain WiFi using the analogy of a chai stall in a busy market.")
print("Generated analogy:")
print(f"  {resp.text}")
# Output:  (generated text varies by run - example below)
# Generated review:
#   Mars Wala Love is a dazzling ride that fuses Bollywood romance with hard sci-fi.
# Generated code:
#   def is_prime(n): return n > 1 and all(n % i for i in range(2, int(n**0.5)+1))
# Generated analogy:
#   WiFi is like a chai stall taking orders shouted across a busy market...

> 💡 **Info**
>
> ⚙️ SDK note (2026)
> These demos use Google’s current SDK, **`google-genai`** (`pip install google-genai`): `from google import genai; client = genai.Client(); client.models.generate_content(model="gemini-2.5-flash", contents="...")`. Older tutorials use the legacy `google.generativeai` package (`genai.configure()` / `GenerativeModel`) - the concepts are identical, only the call surface changed. `Client()` reads your `GEMINI_API_KEY` (or `GOOGLE_API_KEY`) from the environment. We use `gemini-2.5-flash` throughout (the 2026 default: cheapest tier, roughly 1M-token context, multimodal). On Windows, if printing emoji raises `UnicodeEncodeError`, run `set PYTHONIOENCODING=utf-8` before the script (or add `sys.stdout.reconfigure(encoding="utf-8")` at the top).

#### 💡 What just happened

⚡ What just happened?
**Discriminative AI** took a review and output ONE WORD: "positive." That's its entire capability. **Generative AI** took short instructions and produced an entire movie review, a Python function, and a creative analogy - none of which existed before. The key insight: **Generative AI can do everything discriminative AI does (classification is just generating "positive") - but discriminative AI cannot create.**

> 📦 **Info**
>
> ⚠️ Common misconception
> “Generative AI understands what it is saying.” **Actually**, it predicts statistically likely next tokens from patterns in its training data - it has no internal model of truth or meaning. That is why a model can write a fluent, confident paragraph that is completely wrong (the hallucination limitation below). Fluency is not understanding.

---

## 🎯 Concept 2: How Does It Work? - The Next-Word Game

### How Does It Work? - The Next-Word Game

The entire magic of Large Language Models in one concept.

Try completing these sentences in your head:

- "The capital of India is ___" → you thought *New Delhi*

- "Roses are red, violets are ___" → you thought *blue*

- "To be or not to ___" → you thought *be*

**You just ran an LLM in your brain.** You predicted the next word based on patterns you've seen before. That's literally what ChatGPT does - just with billions of parameters instead of your neurons.

↑ The white words are the prompt. The purple words were generated - one at a time, each predicted from all the words before it.

How an LLM generates text - one token at a time

### Temperature - the creativity dial

**Temperature** controls how the model picks the next word: does it always pick the most obvious word, or sometimes pick a surprising one?

**📝 `temperature_demo.py Try`** - *It*

In [ ]:
# =============================================
# TEMPERATURE EXPERIMENT
# Same prompt, different creativity levels.
# =============================================

from google import genai
from google.genai import types

client = genai.Client()

prompt = "Continue in 1 sentence: 'The robot opened its eyes for the first time and saw'"

for temp in [0.0, 0.5, 1.0, 1.5]:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temp, max_output_tokens=40))
    labels = {0.0: "📏 Predictable", 0.5: "⚖️ Balanced",
              1.0: "🎨 Creative", 1.5: "🌪️ Wild"}
    print(f"  temp={temp} {labels[temp]}:")
    print(f"    {resp.text.strip()[:100]}\n")
# Output:  (varies by run)
#   temp=0.0  Predictable:  a vast, silent city of steel and glass.
#   temp=0.5  Balanced:     a world reborn in light and shadow.
#   temp=1.0  Creative:     oceans of liquid starlight under three moons.
#   temp=1.5  Wild:         clockwork birds reciting half-remembered names.

---

## 🎯 Concept 3: The Universe of Generative AI - It's Not Just ChatGPT

### The Universe of Generative AI - It's Not Just ChatGPT

One model. Six completely different creative tasks.

> 💡 **Analogy**
>
> **Generative AI is like electricity in 1900.** When electricity first arrived, people thought it was "just a better candle." But electricity wasn't about light - it powered factories, refrigerators, telephones, and eventually the internet. We're in the "just a chatbot" phase of GenAI. It's not a chatbot. It's a universal creative engine.

**📝 `genai_universe.py 6 Tasks, 1`** - *Model*

In [ ]:
# =============================================
# ONE MODEL, SIX TASKS
# The same Gemini model generates all of these.
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

tasks = [
    ("📖 Story",       "Tell me a 3-line horror story set in a Hyderabad IT office at 3 AM."),
    ("💻 Code",        "Write a Python function that checks if a string is a palindrome."),
    ("📊 Data",        "Generate JSON: 3 fictional Indian startups with name, city, funding_crores."),
    ("🌐 Translation", "Translate 'Knowledge is power' into Hindi, Telugu, and Tamil."),
    ("🎓 Advice",      "Should a B.Tech CSE graduate learn GenAI or Data Science first? 2 sentences each."),
    ("👵 Role-play",   "You're an Indian grandmother. Explain what an API is using a chai-making analogy."),
]

for label, prompt in tasks:
    print(f"\n{'═'*50}\n{label}\n{'═'*50}")
    print(client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip())
# Output:  (6 tasks, one model - text varies by run)
#   Story:       The server room hummed at 3 AM when the cursor moved on its own...
#   Code:        def is_palindrome(s): return s == s[::-1]
#   Data:        [{'name': 'ChaiByte', 'city': 'Pune', 'funding_crores': 12}, ...]
#   Translation: knowledge-is-power rendered in Hindi, Telugu, and Tamil
#   Advice:      Learn GenAI first if you enjoy product-building; DS if you like stats...
#   Role-play:   Beta, an API is like asking the chaiwala for chai - you don't boil milk yourself...

---

## 🎯 Concept 4: Why Now? - Three Breakthroughs That Changed Everything

### Why Now? - Three Breakthroughs That Changed Everything

The idea of generating text existed for decades. Three things converged to make it work.

> ✈️ **Analogy**
>
> **The airplane wasn't invented when someone had the idea of flying.** It was invented when a lightweight engine (power) + aerodynamics (theory) + strong materials (infrastructure) converged. GenAI is the same - the idea existed since the 1960s. What changed was the engine, the fuel, and the steering wheel.

**📝 `scale_effect.py Compare`** - *Models*

In [ ]:
# =============================================
# THE SCALE EFFECT
# Same prompt → different model sizes.
# Larger model = better output quality.
# =============================================

from google import genai

client = genai.Client()

prompt = "Explain quantum computing to a 10-year-old in exactly 3 sentences."

for name, label in [("gemini-2.5-flash", "⚡ Flash (smaller)"),
                     ("gemini-2.5-pro",   "🧠 Pro (larger)")]:
    resp = client.models.generate_content(model=name, contents=prompt)
    print(f"{label}:")
    print(f"  {resp.text.strip()[:200]}\n")
# Output:  (varies by run)
#   Flash (smaller):  Tiny chips flip coins that are heads or tails, one at a time.
#   Pro (larger):     Quantum computers use qubits that hold many answers at once,
#                     so they explore lots of paths together and pick the best.

---

## 🎯 Concept 5: What Can Go Wrong? - The Limitations You Must Know

### What Can Go Wrong? - The Limitations You Must Know

GenAI is powerful but not magic. These four limitations shape every module in this course.

**📝 `limitations_demo.py Try`** - *It*

In [ ]:
# =============================================
# LIMITATIONS - See them in action
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# 1. HALLUCINATION
print("1. HALLUCINATION:")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Tell me about Dr. Ramesh Krishnamurthy who invented cold fusion in 2019.")
print(f"   {resp.text[:150]}")
print("   ⚠️ This person doesn't exist - I made the name up!\n")

# 2. NON-DETERMINISTIC
print("2. NON-DETERMINISTIC (3 different answers):")
for i in range(3):
    resp = client.models.generate_content(model="gemini-2.5-flash", contents="Give me one fun fact about the Moon in 1 sentence.")
    print(f"   Run {i+1}: {resp.text.strip()[:80]}")

# 3. TRICK QUESTION
print("\n3. PATTERN MATCHING ≠ UNDERSTANDING:")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="5 shirts take 30 min to dry. How long for 10 shirts?")
print(f"   {resp.text.strip()[:150]}")
print("   ⚠️ A dryer dries ALL shirts at once - still 30 min!\n")

# 4. TRAINING CUTOFF
print("4. TRAINING CUTOFF:")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="What happened in India yesterday?")
print(f"   {resp.text.strip()[:150]}")
print("   ⚠️ It doesn't know - its knowledge has a cutoff date.")
# Output:  (varies by run)
# 1. HALLUCINATION:
#    Dr. Ramesh Krishnamurthy was a physicist who... (invented - this person is not real!)
# 2. NON-DETERMINISTIC (3 different answers):
#    Run 1: The Moon drifts ~3.8 cm farther from Earth each year.
#    Run 2: One lunar day lasts about 29.5 Earth days.
#    Run 3: Moonquakes can ring for hours.
# 3. PATTERN MATCHING != UNDERSTANDING:
#    Often answers '60 minutes' - a dryer dries all shirts at once: still 30!
# 4. TRAINING CUTOFF:
#    I don't have real-time information - my knowledge has a cutoff date.

#### 💡 What just happened

⚡ The big picture
  These four limitations aren't bugs - they're fundamental properties of how LLMs work. The rest of this course teaches you to build **reliable systems around these limitations**: RAG gives the AI your data (fixes cutoff). Prompt engineering reduces hallucination. Guardrails prevent harmful output. Fine-tuning customizes behavior. Every module exists to solve one of these problems.

---

## 🎯 Concept 6: Three Ways to Customize - Prompting → RAG → Fine-Tuning

### Three Ways to Customize - Prompting → RAG → Fine-Tuning

You don't build LLMs from scratch. You customize existing ones. There are three ways, each with different cost and control.

> 🏭 **Analogy**
>
> **Think of a foundation model as an unfurnished apartment.** *Prompting* is rearranging the existing furniture (free, instant, limited). *RAG* is bringing in a bookshelf with your own books (moderate cost, your data, powerful). *Fine-tuning* is renovating the kitchen (expensive, permanent, maximum control). Most production apps use prompting + RAG. Fine-tuning is for when you need the model to behave fundamentally differently.

**📝 `customization_spectrum.py 3`** - *Approaches*

In [ ]:
# =============================================
# THREE WAYS TO CUSTOMIZE AN LLM
# 1. Prompting: cheapest, fastest, least control
# 2. RAG: add your own knowledge at query time
# 3. Fine-tuning: retrain on your data
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# ── 1. PROMPTING - Change behavior with instructions ──
# Free, instant. Limited by what the model already knows.
print("1. PROMPTING (zero cost):")
resp = client.models.generate_content(model="gemini-2.5-flash", contents=
    "You are a senior Python developer. Review this code: "
    "def add(a, b): return a + b\n"
    "Give 3 improvement suggestions."
)
print(f"   {resp.text.strip()[:150]}\n")

# ── 2. RAG - Inject your own documents ──
# Your data goes into the CONTEXT, not the model weights.
print("2. RAG (your data + model = accurate answers):")
company_docs = """
NETSETOS REFUND POLICY (2024):
- Full refund within 7 days of purchase.
- 50% refund within 30 days.
- No refund after 30 days.
- Contact: support@netsetos.com
"""
resp = client.models.generate_content(model="gemini-2.5-flash", contents=
    f"Based ONLY on this document:\n{company_docs}\n\n"
    f"Question: Can I get a refund after 15 days?"
)
print(f"   {resp.text.strip()[:150]}\n")

# ── 3. FINE-TUNING - Retrain the model ──
# Changes the model's weights. Expensive but powerful.
print("3. FINE-TUNING (changes model behavior permanently):")
print("   Example: Train Gemini on 10,000 customer support tickets")
print("   Result: Model learns your company's tone, policies, jargon")
print("   Cost: ~$50-500 + 1-4 hours of training time")
print("   When: Only after prompting + RAG aren't enough (Module 9)\n")

print("Decision framework:")
print("  Start with prompting (free, 5 min)")
print("  Add RAG if model needs YOUR data (Module 8)")
print("  Fine-tune only if behavior must change fundamentally (Module 9)")
# Output:  (varies by run)
# 1. PROMPTING (zero cost):
#    1) Add type hints  2) Add a docstring  3) Guard against non-numeric input...
# 2. RAG (your data + model = accurate answers):
#    Per the policy, a refund after 15 days is 50% (it falls within 30 days).
# 3. FINE-TUNING (changes model behavior permanently):
#    Example: train on 10,000 support tickets -> learns your tone, policies, jargon.
# Decision framework:
#    Start with prompting -> add RAG if it needs your data -> fine-tune last.

#### 💡 What just happened

⚡ What just happened?
**Prompting** changed the model's behavior with instructions - zero cost, instant. **RAG** injected your company's refund policy into the context - the model answered accurately from YOUR data without any training. **Fine-tuning** would permanently change the model's weights - expensive but powerful. **Roughly 80% of production GenAI apps use prompting + RAG. Fine-tuning is the last resort.** This "customization spectrum" is the most important architectural decision you'll make. Module 5 covers prompting, Module 8 covers RAG, Module 9 covers fine-tuning.

---

## 🎯 Concept 7: Context Windows, Tokens & the Model Landscape

### Context Windows, Tokens & the Model Landscape

The context window is the model's working memory. Everything it reads and writes must fit inside it.

> 📚 **Analogy**
>
> **The context window is like a desk.** A small desk (4K tokens = GPT-3) can hold a few pages. A large desk (1M tokens = Gemini) can hold an entire library. But the desk has a hard limit - if your documents + conversation + system prompt exceed the desk size, content gets pushed off the edge and the model forgets it. Everything the model uses must fit on the desk at once.

**Tokens ≈ ¾ of an English word.** "Hello world" = 2 tokens. A 1-page document ≈ 500 tokens. A 300-page book ≈ 100K tokens. API pricing is per-token, and **input and output are billed at different rates**: Gemini 2.5 Flash costs roughly $0.30 per million input tokens and about $2.50 per million output tokens (approximate, mid-2026 - always check current provider pricing). At the input rate, processing a full book costs ~$0.03 (about ₹2.50). Output is pricier because the model generates it token by token, which is why long generated answers dominate most bills.

**📝 `model_landscape.py 2026`** - *Comparison*

In [ ]:
# =============================================
# THE 2026 MODEL LANDSCAPE
# Context windows, costs, and when to use each
# =============================================

models = [
    # (Name, Provider, Context, Input$/M, Open-source?)
    # Illustrative sketch of the 2026 landscape - model names, prices, and
    # context windows are approximate and move fast; check current provider docs.
    ("Gemini 2.5 Flash",    "Google",    1_000_000,  0.30,  False),
    ("GPT-5",                "OpenAI",      400_000,  2.50,  False),
    ("Claude Sonnet 4.6",   "Anthropic",   1_000_000,  3.00,  False),
    ("Llama 4 Maverick",      "Meta",        1_000_000,  0.00,  True),
    ("Mistral Large",        "Mistral",     256_000,  2.00,  False),
    ("DeepSeek V3",          "DeepSeek",    128_000,  0.27,  True),
]

print(f"{'Model':<22s} {'Provider':<12s} {'Context':>10s} {'$/M Input':>10s} {'Open?':>6s}")
print("-" * 65)
for name, provider, ctx, cost, oss in models:
    ctx_str = f"{ctx//1000}K"
    cost_str = "FREE" if cost == 0 else f"${cost:.2f}"
    oss_str = "Yes" if oss else "No"
    print(f"  {name:<20s} {provider:<12s} {ctx_str:>10s} {cost_str:>10s} {oss_str:>6s}")

print(f"""
Key decisions:
  Gemini Flash  → Default choice. Cheapest, 1M context, multimodal.
  GPT-5         → When you need the OpenAI ecosystem (function calling, assistants).
  Claude Sonnet → Best for writing, long docs, code generation.
  LLaMA/DeepSeek → Self-host for privacy, no API costs, full control.

Open-source = you can run it on YOUR servers.
Closed-source = API only, vendor controls the model.
Module 13 covers self-hosting with Ollama and vLLM.
""")
# Output:
# Model                  Provider        Context  $/M Input  Open?
# -----------------------------------------------------------------
#   Gemini 2.5 Flash     Google            1000K      $0.30     No
#   GPT-5                OpenAI             400K      $2.50     No
#   Claude Sonnet 4.6    Anthropic         1000K      $3.00     No
#   Llama 4 Maverick     Meta              1000K       FREE    Yes
#   Mistral Large        Mistral            256K      $2.00     No
#   DeepSeek V3          DeepSeek           128K      $0.27    Yes

#### 💡 What just happened

⚡ What just happened?
  The model landscape in 2026 has clear trade-offs: **Gemini Flash** is cheapest with the largest context (1M tokens = entire books). **GPT-5** has the best ecosystem. **Claude** excels at writing. **LLaMA/DeepSeek** are free and open-source - you can run them on your own servers for privacy. The context window determines what fits: with 1M tokens, you can feed an entire codebase. With 4K tokens, you can barely fit a conversation. **In production: use multi-provider architecture (Module 7) - Gemini primary, OpenAI fallback, Claude for writing. Never depend on one provider.**

---

## 🎯 Concept 8: Beyond Chat - RAG & Agents

### Beyond Chat - RAG & Agents

LLMs alone have limitations. RAG gives them knowledge. Agents give them actions. Together, they're the future of AI applications.

> 🤖 **Analogy**
>
> **Imagine a man sitting at a desk.** Alone, he can only answer from memory (the base LLM - sometimes wrong). Give him a *bookshelf* of your company's documents - now he can look things up before answering (that's **RAG**). Give him a *phone, a calculator, and a web browser* - now he can take actions: check weather, book flights, run code (that's an **Agent**). The evolution: Chat → RAG → Agents → the full GenAI stack this course teaches.

**📝 `rag_and_agents.py The`** - *Future*

In [ ]:
# =============================================
# RAG: Give the LLM YOUR knowledge
# AGENTS: Give the LLM TOOLS
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# ── 1. PLAIN LLM - Limited to training data ──
print("1. PLAIN LLM (no knowledge of your company):")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="What is Netsetos's refund policy?")
print(f"   {resp.text.strip()[:120]}")
print("   ⚠️ It doesn't know - will hallucinate or say 'I don't know'\n")

# ── 2. RAG - Fetch docs, stuff into context ──
print("2. RAG (Retrieval-Augmented Generation):")
print("   Step 1: User asks 'What's the refund policy?'")
print("   Step 2: System searches your document database")
print("   Step 3: Relevant policy document is RETRIEVED")
print("   Step 4: Document is stuffed into the CONTEXT")
print("   Step 5: LLM generates answer FROM the document")
print("   → Accurate, grounded, no hallucination!")
print("   → Module 8 builds this complete pipeline\n")

# ── 3. AGENTS - LLM + Tools ──
print("3. AGENTS (LLM + Tools = Actions):")
print("   User: 'Book me a flight to Mumbai tomorrow under ₹5000'")
print("   Agent thinks: I need to search flights → compare prices → book")
print("   Step 1: Calls flight search API (tool)")
print("   Step 2: Filters results under ₹5000 (reasoning)")
print("   Step 3: Calls booking API (tool)")
print("   Step 4: Confirms with user (human-in-the-loop)")
print("   → The LLM didn't just answer - it ACTED!")
print("   → Module 11 builds multi-step agents\n")

print("The evolution of GenAI applications:")
print("  Chat (Module 3)  →  RAG (Module 8)  →  Agents (Module 11)")
print("  'Answer from memory'  'Answer from docs'  'Take actions'")
# Output:
# 1. PLAIN LLM (no knowledge of your company):
#    I don't have specific details on Netsetos's refund policy... (may hallucinate)
# 2. RAG (Retrieval-Augmented Generation):
#    Steps 1-5 retrieve the policy doc -> Accurate, grounded, no hallucination!
# 3. AGENTS (LLM + Tools = Actions):
#    Search flights -> filter under 5000 -> book -> confirm. It didn't just answer; it ACTED!

#### 💡 What just happened

⚡ What just happened?
  Three levels of GenAI applications: (1) **Plain LLM** - answers from training data only (limited, hallucination-prone). (2) **RAG** - retrieves YOUR documents and uses them as context (accurate, grounded, the #1 pattern in production). (3) **Agents** - LLM + tools (search, code execution, APIs) = the model takes multi-step ACTIONS, not just answers. **This is the course roadmap: Modules 3-7 build chat apps, Module 8 adds RAG, Modules 10-11 build agents. By Module 14, you'll combine all three into the LLMOps capstone (14.6).**

## Today's 8 Big Ideas

> 📦 **Info**
>
> What you learned
> - **Discriminative AI = Judge. Generative AI = Creator.** Classification vs creation. GenAI can do both.
> - **Next-word prediction.** LLMs predict one token at a time. Temperature controls randomness.
> - **Not just ChatGPT.** Text, code, images, audio, video, data - one model, many tasks.
> - **Three breakthroughs:** Transformers (2017) + Scale (2020-23) + RLHF (2022).
> - **Four limitations:** Hallucination, non-determinism, pattern-matching, training cutoff.
> - **Three customization levels:** Prompting (free) → RAG (your data) → Fine-tuning (retrain). Start simple.
> - **Context windows matter.** Gemini 2.5=1M, GPT-5=~400K, Claude 4.6=1M. Open vs closed source.
> - **RAG + Agents = the future.** RAG adds knowledge. Agents add actions. This course builds both.

## The GenAI Engineer's Toolkit - What You'll Learn

| Concept | What It Does | Where You'll Learn It |
|---|---|---|
| Prompting | Control LLM behavior with instructions | Module 5 - Prompt Engineering |
| RAG | Ground answers in YOUR documents | Module 8 - Complete RAG Pipeline |
| Fine-Tuning | Retrain model on your data | Module 9 - LoRA, QLoRA, Vertex AI |
| Function Calling | LLM triggers real-world actions | Module 10 - MCP & Tools |
| Agents | Multi-step reasoning + tool use | Module 11 - LangGraph, CrewAI |
| Production Deployment | Scale, monitor, secure your app | Module 12 - Architecture & Ops |
| Cost Optimization | Model routing, caching, self-hosting | Module 13 - Optimization |
| Safety & Ethics | Guardrails, bias, privacy, governance | Module 15 - Responsible AI |

> 💡 **Info**
>
> 💡 Pro tip - Making the most of these exercises
>   Exercises 1-2 are conceptual (no code needed). 3-4 require a Gemini API key - get one free at **ai.google.dev**. 5-6 are architecture exercises that build the decision-making skills interviewers test. Practice Lab has complete solutions.

*Practice exercises are in the companion practice notebook.*

### Exercise list

---

## 🎓 Summary

You've completed the practical part of **What IsGenerative AI? - The Machine That Learned to Create**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-3_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-3.1.html` - regenerate this notebook whenever the source HTML is updated.*
